In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
torch.cuda.is_available()

True

# Generate a mock dataset

In [3]:
import random
random.seed(55)
# why do we need this part? Can we make it more efficient or make it in pandas dataframe in one go?
CATALOG = {
    1: ("SmartTerm 20", "Term"),
    2: ("Term100 Protect", "Term"),
    3: ("LegacyGold Whole Life", "WholeLife"),
    4: ("Prestige Whole Life", "WholeLife"),
    5: ("WealthBuilder Endowment", "Endowment"),
    6: ("SavingsPlan Endowment", "Endowment"),
    7: ("FlexiInvest Linked", "ILP"),
    8: ("CI Shield", "CriticalIllness"),
}

ITEM_IDS_BY_CAT = {}
for item_id, (_, cat) in CATALOG.items():
    ITEM_IDS_BY_CAT.setdefault(cat, []).append(item_id)

ITEM_IDS_BY_CAT

{'Term': [1, 2],
 'WholeLife': [3, 4],
 'Endowment': [5, 6],
 'ILP': [7],
 'CriticalIllness': [8]}

In [4]:
COVERAGE_MULTIPLE = {
    "Term": (150, 300),
    "WholeLife": (40, 80),
    "Endowment": (3, 8),
    "ILP": (10, 30),
    "CriticalIllness": (50, 100),
}

In [5]:
BASE_COVERAGE = {
    "Term": 1.5,
    "WholeLife": 1.0,
    "Endowment": 0.4,
    "ILP": 0.6,
    "CriticalIllness": 0.3
}

In [6]:
def age_factor(age:float, factor_a:int=25, factor_b:int=45, offset:float=0.8)->float:
    return 1.0 + max(0.0, (age-factor_a)/factor_b) * offset

In [7]:
def sample_category(age: float)->str:
    weights = {
        "Term": max(0.1, 1.5-age/40),
        "CriticalIllness": max(0.1, 1.2-age/50),
        "WholeLife": min(1.5, age/35),
        "Endowment": min(1.3, age/45),
        "ILP": 0.7
    }
    cats = list(weights.keys())
    probs = list(weights.values())
    return random.choices(cats, weights=probs, k=1)[0]

In [8]:
def sample_item_in_category(cat: str)->int:
    return random.choice(ITEM_IDS_BY_CAT[cat])

In [9]:
def lognormal_noise(sigma: float=0.15)->float:
    return float(torch.exp(torch.randn(1) *sigma))

In [10]:
def generate_customer(customer_id: int, n_events: int | None = None, max_events:int = 8):
    if n_events is None:
        n_events = random.randint(2, max_events)
    wealth = float(torch.exp(torch.randn(1) * 0.5 + 2.0))
    age = float(random.randint(22, 55))
    items, ages, prices, sum_insures = [], [], [], []
    for _ in range(n_events):
        cat = sample_category(age)
        item_id = sample_item_in_category(cat)
        low, high = COVERAGE_MULTIPLE[cat]
        multiple = random.uniform(low, high)
        sum_insure = wealth * BASE_COVERAGE[cat] * 10_000 * lognormal_noise()
        price = sum_insure / (multiple * age_factor(age)) * lognormal_noise()

        items.append(item_id)
        ages.append(age)
        prices.append(price)
        sum_insures.append(sum_insure)

        age += random.randint(1, 5)
    return items, ages, prices, sum_insures

In [11]:
def generate_dataset(n_customers: int = 500):
    return [generate_customer(customer_id=i) for i in range(n_customers)]

In [12]:
items, ages, prices, sum_insures = generate_customer(customer_id=1, n_events=6)

for item_id, age, price, sum_insure in zip(items, ages, prices, sum_insures):
    name, cat = CATALOG[item_id]
    ratio = sum_insure / price
    print(f"age {age:>4.0f} | {name:<24} ({cat:<15}) | "
            f"price {price:>10,.0f} | sum_insure {sum_insure:>12,.0f} | ratio {ratio:>6.1f}x")

age   27 | FlexiInvest Linked       (ILP            ) | price      2,999 | sum_insure       38,323 | ratio   12.8x
age   30 | FlexiInvest Linked       (ILP            ) | price      2,781 | sum_insure       45,457 | ratio   16.3x
age   33 | FlexiInvest Linked       (ILP            ) | price      1,774 | sum_insure       48,013 | ratio   27.1x
age   37 | Prestige Whole Life      (WholeLife      ) | price      1,011 | sum_insure       88,800 | ratio   87.8x
age   41 | FlexiInvest Linked       (ILP            ) | price      2,138 | sum_insure       50,852 | ratio   23.8x
age   42 | FlexiInvest Linked       (ILP            ) | price      1,376 | sum_insure       47,202 | ratio   34.3x


# Construct it into pytorch dataset and dataloader

In [13]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split

MAX_EVENTS = 8

def pad(seq, max_len, pad_value:int | float =0):
    return seq[:max_len] + ([pad_value] * max(0, max_len-len(seq)))

In [14]:
class TransactionDataset(Dataset):
    """Return items, ages, prices, sum_insures"""
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        items, ages, prices, sum_insures = self.samples[idx]
        return (
            torch.tensor(pad(items, MAX_EVENTS), dtype=torch.long),
            torch.tensor(pad(ages, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(-1),
            torch.tensor(pad(prices, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(-1),
            torch.tensor(pad(sum_insures, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(-1),
        )

In [15]:
samples = generate_dataset(n_customers=500)
dataset = TransactionDataset(samples)

n_total = len(dataset)
n_train = int(n_total * .7)
n_val = int(n_total * .15)
n_test = n_total - n_train - n_val

train_set, val_set, test_set = random_split(
    # this approach can be used with Dataset subclass?
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(55)
)

len(train_set), len(val_set), len(test_set)

(350, 75, 75)

In [16]:
BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

In [17]:
class Normalizer:
    def __init__(self):
        self.mean:float | None = None
        self.std:float | None = None

    def fit(self, values: torch.Tensor):
        self.mean = values.mean().item()
        self.std = values.std().item()

    def transform(self, values: torch.Tensor) -> torch.Tensor:
        return (values - self.mean) / self.std

    def inverse_transform(self, values: torch.Tensor) -> torch.Tensor:
        return (values * self.std) + self.mean

In [18]:
def collect_train_values(base_dataset, train_subset, field_idx):
    values = []
    for i in train_subset.indices:
        values.extend(base_dataset.samples[i][field_idx])
    return torch.tensor(values, dtype=torch.float)

In [19]:
age_normalizer = Normalizer()
price_normalizer = Normalizer()
sum_insure_normalizer = Normalizer()

In [20]:
age_normalizer.fit(collect_train_values(dataset, train_set, field_idx=1))
price_normalizer.fit(collect_train_values(dataset, train_set, field_idx=2))
sum_insure_normalizer.fit(collect_train_values(dataset, train_set, field_idx=3))

In [21]:
items, ages, prices, sum_insures = next(iter(train_loader))
print(items.shape, ages.shape, prices.shape, sum_insures.shape)

ages_norm = age_normalizer.transform(ages)
print(f"normalized age batch mean ≈ {ages_norm.mean().item():.2f} (should be near 0, not exactly)")

torch.Size([32, 8]) torch.Size([32, 8, 1]) torch.Size([32, 8, 1]) torch.Size([32, 8, 1])
normalized age batch mean ≈ -1.79 (should be near 0, not exactly)


# Fusion design

In [22]:
import torch
import torch.nn as nn
from typing import Literal

class FusionEmbedding(nn.Module):
    def __init__(self, categorical_vocab_sizes:list[int], num_continuous: int, d_model:int, strategy:Literal['sum', 'concat_all', 'concat_emb_summed'] = "sum"):
        super().__init__()
        assert strategy in ("sum", "concat_all", "concat_emb_summed")
        self.strategy = strategy
        self.num_categorical = len(categorical_vocab_sizes)
        self.num_continuous = num_continuous
        self.categorical_embs = nn.ModuleList([
            nn.Embedding(vocab_size+1, d_model, padding_idx=0)
            for vocab_size in categorical_vocab_sizes
        ])
        self.continuous_projs = nn.ModuleList([
            nn.Linear(1, d_model)
            for _ in range(num_continuous)
        ])
        if strategy == "concat_all":
            concat_dim = d_model * (self.num_categorical + self.num_continuous)
            self.project_down = nn.Linear(concat_dim, d_model)

        elif strategy == "concat_emb_summed":
            concat_dim = d_model*2
            self.project_down = nn.Linear(concat_dim, d_model)

        else:
            self.project_down = None
        
    def forward(self, categorical_features: list[torch.Tensor], continuous_features: list[torch.Tensor]) -> torch.Tensor:
        assert len(categorical_features) == self.num_categorical
        assert len(continuous_features) == self.num_continuous

        cat_vecs = [
            emb(feat) for emb, feat in zip(self.categorical_embs, categorical_features)
        ]

        cont_vecs = [
            emb(feat) for emb, feat in zip(self.continuous_projs, continuous_features)
        ]
        
        if self.strategy == "sum":
            return sum(cat_vecs) + sum(cont_vecs)
        
        if self.strategy == "concat_all":
            fused = torch.cat(cat_vecs + cont_vecs, dim=-1)
            return self.project_down(fused)
        
        fused = torch.cat([sum(cat_vecs), sum(cont_vecs)], dim=-1)
        return self.project_down(fused)

In [23]:
batch, seq_len, d_model = 4, 8, 32
categorical_vocab_sizes = [8, 5]   # e.g. item_id (8 items), sales_channel (5 channels)
num_continuous = 3                  # age, price, sum_insure

item_id    = torch.randint(0, categorical_vocab_sizes[0] + 1, (batch, seq_len))
channel_id = torch.randint(0, categorical_vocab_sizes[1] + 1, (batch, seq_len))
age        = torch.randn(batch, seq_len, 1)
price      = torch.randn(batch, seq_len, 1)
sum_insure = torch.randn(batch, seq_len, 1)

for strategy in ["sum", "concat_all", "concat_emb_summed"]:
    fusion = FusionEmbedding(categorical_vocab_sizes, num_continuous, d_model, strategy=strategy)
    out = fusion([item_id, channel_id], [age, price, sum_insure])
    print(f"{strategy:<20} -> {out.shape}")   # expect (4, 8, 32) for every strategy

sum                  -> torch.Size([4, 8, 32])
concat_all           -> torch.Size([4, 8, 32])
concat_emb_summed    -> torch.Size([4, 8, 32])


In [24]:
test_emb = nn.Embedding(10, 10)
test_lin = nn.Linear(1, 10)
items = torch.tensor([1,2,3], dtype=torch.long)
ages = torch.tensor([25, 30, 45], dtype=torch.float).unsqueeze(-1)

In [25]:
iemb = test_emb(items)
alin = test_lin(ages)

In [26]:
iemb.shape, alin.shape

(torch.Size([3, 10]), torch.Size([3, 10]))

In [27]:
(iemb+alin).shape

torch.Size([3, 10])

In [28]:
torch.cat([iemb, alin], dim=-1).shape

torch.Size([3, 20])

# Define loss

In [29]:
import torch.nn as nn

class UncertaintyWeightedLoss(nn.Module):
    def __init__(self, task_types: list[str]):
        super().__init__()
        self.task_types = task_types
        self.log_vars = nn.Parameter(torch.zeros(len(task_types)))

    def forward(self, losses: list[torch.Tensor]) -> torch.Tensor:
        total = torch.tensor(0.0)
        for i, (loss, task_type) in enumerate(zip(losses, self.task_types)):
            precision = torch.exp(-self.log_vars[i]) 
            if task_type == "classification":
                total += (precision * loss) + self.log_vars[i]
            else:
                total += (precision * loss) + (0.5 * self.log_vars[i])
            
        return total

    def precisions(self):
        return {i: torch.exp(-self.log_vars[i]).item() for i in range(len(self.task_types))}

# Encoder Block

## Setup

In [30]:
import torch
import torch.nn as nn
from package.encoder_block import EncoderBlock

class EncoderCLSBackbone(nn.Module):
    def __init__(self, fusion, d_model: int, n_heads: int, max_len: int, num_layers: int =2):
        super().__init__()
        self.fusion = fusion
        self.cls = nn.Parameter(torch.randn(1, 1, d_model))
        self.layers = nn.ModuleList([
            # max_len must be added 1 because we will implement cls prepending
            EncoderBlock(d_model, n_heads, max_len+1) for _ in range(num_layers)
        ])
    
    def forward(self, categorical_features, continuous_features, pad_mask_source):
        batch = pad_mask_source.shape[0]
        pad_mask = pad_mask_source == 0
        cls_col = torch.zeros(batch, 1, dtype=torch.bool, device=pad_mask_source.device)
        pad_mask = torch.concat([cls_col, pad_mask], dim=1)
        x = self.fusion(categorical_features, continuous_features)
        cls = self.cls.expand(batch, -1, -1)
        x = torch.cat([cls, x], dim=1)

        for layer in self.layers:
            x = layer(x, pad_mask)
        
        # return only CLS vector at the 1st position of each sequence
        return x[:, 0]

In [31]:
batch, seq_len, d_model, n_heads = 4, 8, 32, 4
num_items, num_continuous = 8, 3

fusion = FusionEmbedding([num_items], num_continuous, d_model, strategy="sum")
backbone = EncoderCLSBackbone(fusion, d_model=d_model, n_heads=n_heads, max_len=seq_len, num_layers=2)

item_id    = torch.randint(0, num_items + 1, (batch, seq_len))
age        = torch.randn(batch, seq_len, 1)
price      = torch.randn(batch, seq_len, 1)
sum_insure = torch.randn(batch, seq_len, 1)

summary = backbone([item_id], [age, price, sum_insure], pad_mask_source=item_id)
print(summary.shape)   # expect (4, 32) — one vector per customer, actually collapsed this time

torch.Size([4, 32])


In [32]:
def expand_into_prefixes(samples):
    expanded = []
    for items, ages, prices, sum_insures in samples:
        n = len(items)
        for prefix_len in range(1, n):
            expanded.append((
                items[:prefix_len],
                ages[:prefix_len],
                prices[:prefix_len],
                sum_insures[:prefix_len],
                items[prefix_len],
                ages[prefix_len],
                prices[prefix_len],
                sum_insures[prefix_len]
            ))
    return expanded

class NextSequenceDataset(Dataset):
    def __init__(self, expanded_samples):
        self.samples = expanded_samples

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure = self.samples[idx]
        return (
            torch.tensor(pad(items, MAX_EVENTS), dtype=torch.long),
            torch.tensor(pad(ages, MAX_EVENTS), dtype=torch.float).unsqueeze(-1),
            torch.tensor(pad(prices, MAX_EVENTS), dtype=torch.float).unsqueeze(-1),
            torch.tensor(pad(sum_insures, MAX_EVENTS), dtype=torch.float).unsqueeze(-1),
            torch.tensor(next_item, dtype=torch.long),
            torch.tensor([next_age], dtype=torch.float),
            torch.tensor([next_price], dtype=torch.float),
            torch.tensor([next_sum_insure], dtype=torch.float),
        )

In [33]:
def collect_raw_samples(base_dataset, subset):
    return [base_dataset.samples[i] for i in subset.indices]

train_expanded = expand_into_prefixes(collect_raw_samples(dataset, train_set))
val_expanded = expand_into_prefixes(collect_raw_samples(dataset, val_set))
test_expanded = expand_into_prefixes(collect_raw_samples(dataset, test_set))

In [34]:
next_item_train_loader = DataLoader(NextSequenceDataset(train_expanded), batch_size=BATCH_SIZE, shuffle=True)
next_item_val_loader = DataLoader(NextSequenceDataset(val_expanded), batch_size=BATCH_SIZE, shuffle=False)
next_item_test_loader = DataLoader(NextSequenceDataset(test_expanded), batch_size=BATCH_SIZE, shuffle=False)

print(f"train rows: {len(train_expanded)}  val rows: {len(val_expanded)}  test rows: {len(test_expanded)}")


train rows: 1421  val rows: 276  test rows: 318


In [35]:
class MultiTaskModel(nn.Module):
    def __init__(self, backbone, d_model: int, num_items: int):
        super().__init__()
        self.backbone = backbone
        # num_items + 1 because we reserve 0 for padding
        # item_head will return as logit
        self.item_head = nn.Linear(d_model, num_items+1)
        self.age_head = nn.Linear(d_model, 1)
        self.price_head = nn.Linear(d_model, 1)
        self.sum_insure_head = nn.Linear(d_model, 1)

    def forward(self, categorical_features, continuous_features, pad_mask_source):
        summary = self.backbone(categorical_features, continuous_features, pad_mask_source)
        # is it possible to factor this out into .predict_age_with_constraint?
        ages_norm = continuous_features[0]
        lengths = (pad_mask_source != 0).sum(dim=1)
        batch_idx = torch.arange(ages_norm.shape[0], device=ages_norm.device)
        current_age = ages_norm[batch_idx, lengths-1, 0]
        delta_age = nn.functional.softplus(self.age_head(summary).squeeze(-1))
        age_pred = (current_age + delta_age).unsqueeze(-1)
        return (
            self.item_head(summary),
            age_pred,
            self.price_head(summary),
            self.sum_insure_head(summary),
        )

In [36]:
fusion   = FusionEmbedding([num_items], num_continuous=3, d_model=32, strategy="sum")
backbone = EncoderCLSBackbone(fusion, d_model=32, n_heads=4, max_len=MAX_EVENTS, num_layers=2)
model    = MultiTaskModel(backbone, d_model=32, num_items=num_items)

items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure = next(iter(next_item_train_loader))

# normalize every continuous value — both the input sequence AND the regression
# labels — using the SAME normalizers already fit on the train split only
ages_norm             = age_normalizer.transform(ages)
prices_norm           = price_normalizer.transform(prices)
sum_insures_norm      = sum_insure_normalizer.transform(sum_insures)
next_age_norm         = age_normalizer.transform(next_age)
next_price_norm       = price_normalizer.transform(next_price)
next_sum_insure_norm  = sum_insure_normalizer.transform(next_sum_insure)

item_logits, age_pred, price_pred, sum_insure_pred = model(
    [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
)

loss_item       = nn.functional.cross_entropy(item_logits, next_item)
loss_age        = nn.functional.mse_loss(age_pred, next_age_norm)
loss_price      = nn.functional.mse_loss(price_pred, next_price_norm)
loss_sum_insure = nn.functional.mse_loss(sum_insure_pred, next_sum_insure_norm)

print(loss_item.item(), loss_age.item(), loss_price.item(), loss_sum_insure.item())


2.3356752395629883 1.637583613395691 1.6023436784744263 2.493215799331665


In [37]:
loss_weigher = UncertaintyWeightedLoss(task_types=["classification", "regression", "regression", "regression"])

total_loss = loss_weigher([loss_item, loss_age, loss_price, loss_sum_insure])
print(total_loss.item())

8.068818092346191


In [38]:
items.shape, item_logits.shape

(torch.Size([32, 8]), torch.Size([32, 9]))

In [39]:
items[0], (item_logits[0])

(tensor([7, 4, 7, 0, 0, 0, 0, 0]),
 tensor([-0.9697,  0.8172,  0.0198,  1.2287,  0.9965, -0.0166, -0.3667,  0.5408,
          0.9166], grad_fn=<SelectBackward0>))

In [40]:
next_item[0], torch.argmax(item_logits[0])

(tensor(2), tensor(3))

In [41]:
ages[0].squeeze(-1), age_pred[0], age_normalizer.inverse_transform(age_pred[0]), next_age[0]

(tensor([27., 31., 34.,  0.,  0.,  0.,  0.,  0.]),
 tensor([0.5729], grad_fn=<SelectBackward0>),
 tensor([52.1131], grad_fn=<AddBackward0>),
 tensor([37.]))

## Test train loop

In [42]:
D_MODEL = 32
N_HEADS = 4
NUM_LAYERS = 2

fusion = FusionEmbedding([num_items], num_continuous=3, d_model=D_MODEL, strategy="sum")
backbone = EncoderCLSBackbone(fusion, d_model=D_MODEL, n_heads=N_HEADS, max_len=MAX_EVENTS, num_layers=NUM_LAYERS)
model = MultiTaskModel(backbone, d_model=D_MODEL, num_items=num_items)
loss_weigher = UncertaintyWeightedLoss(task_types=["classification", "regression", "regression", "regression"])

optimizer = torch.optim.Adam(list(model.parameters())+list(loss_weigher.parameters()), lr=1e-3)
EPOCHS = 50

def normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure):
    return(
        age_normalizer.transform(ages),
        price_normalizer.transform(prices),
        sum_insure_normalizer.transform(sum_insures),
        age_normalizer.transform(next_age),
        price_normalizer.transform(next_price),
        sum_insure_normalizer.transform(next_sum_insure),
    )

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure in next_item_train_loader:
        ages_norm, prices_norm, sum_insures_norm, next_age_norm, next_price_norm, next_sum_insure_norm = (
            normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure)
        )
        item_logits, age_pred, price_pred, sum_insure_pred = model(
            [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
        )

        loss_item = nn.functional.cross_entropy(item_logits, next_item)
        loss_age = nn.functional.mse_loss(age_pred, next_age_norm)
        loss_price = nn.functional.mse_loss(price_pred, next_price_norm)
        loss_sum_insure = nn.functional.mse_loss(sum_insure_pred, next_sum_insure_norm)

        loss = loss_weigher([loss_item, loss_age, loss_price, loss_sum_insure])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch+1) % 5 == 0:
        weights = torch.exp(-loss_weigher.log_vars).detach()
        print(
            f"epoch {epoch+1:3d} | train loss {total_loss / len(next_item_train_loader):4f} "
            f"| weights item={weights[0]:.2f} age={weights[1]:.2f} price={weights[2]:.2f} sum_insure={weights[3]:.2f}"
        )

        model.eval()
        correct, total = 0, 0
        age_abs_err, price_abs_err, sum_insure_abs_err = 0.0, 0.0, 0.0

        with torch.no_grad():
            for items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure in next_item_val_loader:
                ages_norm, prices_norm, sum_insures_norm, *_ = (
                    normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure)
                )

                item_logits, age_pred, price_pred, sum_insure_pred = model(
                    [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
                )

                correct += (item_logits.argmax(dim=-1) == next_item).sum().item()
                total += next_item.numel()

                age_abs_err += (age_normalizer.inverse_transform(age_pred) - next_age).abs().sum().item()
                price_abs_err += (price_normalizer.inverse_transform(price_pred) - next_price).abs().sum().item()
                sum_insure_abs_err += (sum_insure_normalizer.inverse_transform(sum_insure_pred) - next_sum_insure).abs().sum().item()
        n_val = len(next_item_val_loader.dataset)
        print(
            f"    |- val item_acc {correct/total:.2%} "
            f"| age MAE {age_abs_err/n_val:.2f} | price MAE {price_abs_err/n_val:.2f} | sum_insure MAE {sum_insure_abs_err/n_val:.2f}"
        )
        model.train()
    

epoch   5 | train loss 3.016730 | weights item=0.83 age=1.26 price=0.91 sum_insure=0.94
    |- val item_acc 16.30% | age MAE 1.31 | price MAE 1482.01 | sum_insure MAE 28137.77
epoch  10 | train loss 2.757526 | weights item=0.72 age=1.57 price=0.85 sum_insure=0.93
    |- val item_acc 16.30% | age MAE 1.28 | price MAE 1409.39 | sum_insure MAE 29037.67
epoch  15 | train loss 2.550005 | weights item=0.64 age=1.96 price=0.81 sum_insure=0.93
    |- val item_acc 15.58% | age MAE 1.29 | price MAE 1426.32 | sum_insure MAE 29724.56
epoch  20 | train loss 2.332727 | weights item=0.59 age=2.45 price=0.79 sum_insure=0.96
    |- val item_acc 17.39% | age MAE 1.29 | price MAE 1391.46 | sum_insure MAE 31485.95
epoch  25 | train loss 2.087194 | weights item=0.57 age=3.05 price=0.81 sum_insure=1.02
    |- val item_acc 17.75% | age MAE 1.31 | price MAE 1478.31 | sum_insure MAE 30051.78
epoch  30 | train loss 1.851288 | weights item=0.56 age=3.78 price=0.86 sum_insure=1.11
    |- val item_acc 15.94% | age

In [43]:
def theoretical_item_ceiling(age: float) -> float:
    """Bayes-optimal probability of guessing the exact next item correctly,
    using the generator's own category weights — the best any model can do."""
    weights = {
        "Term":            max(0.1, 1.5 - age / 40),
        "CriticalIllness": max(0.1, 1.2 - age / 50),
        "WholeLife":       min(1.5, age / 35),
        "Endowment":       min(1.3, age / 45),
        "ILP":             0.7,
    }
    total = sum(weights.values())
    best_cat, best_cat_weight = max(weights.items(), key=lambda kv: kv[1])

    cat_prob            = best_cat_weight / total                    # P(best category | age)
    item_prob_given_cat = 1 / len(ITEM_IDS_BY_CAT[best_cat])          # uniform pick within that category
    return cat_prob * item_prob_given_cat


# average over the actual ages seen in validation — each val_expanded row is
# (items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure)
current_ages = [ages[-1] for (items, ages, prices, sum_insures, *_ ) in val_expanded]
ceiling = sum(theoretical_item_ceiling(age) for age in current_ages) / len(current_ages)

print(f"theoretical best possible next-item accuracy: {ceiling:.1%}")

theoretical best possible next-item accuracy: 16.8%


## Refactor training process

In [44]:
def build_model(
        architecture: str, 
        fusion_strategy: Literal["sum", "concat_all", "concat_emb_summed"],
        num_items: int,
        num_continuous: int = 3,
        d_model: int = 32,
        n_heads: int = 4,
        num_layers: int = 2,
        max_len: int = 8
):
    fusion = FusionEmbedding([num_items], num_continuous, d_model, strategy=fusion_strategy)

    if architecture == 'encoder':
        backbone = EncoderCLSBackbone(fusion, d_model=d_model, n_heads=n_heads, max_len=max_len, num_layers=num_layers)
    else:
        raise ValueError(f"unknown architecture: {architecture}")

    model = MultiTaskModel(backbone, d_model, num_items)
    loss_weigher = UncertaintyWeightedLoss(task_types=["classification", "regression", "regression", "regression"])
    return model, loss_weigher

In [45]:
def normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure):
    return (
        age_normalizer.transform(ages),
        price_normalizer.transform(prices),
        sum_insure_normalizer.transform(sum_insures),
        age_normalizer.transform(next_age),
        price_normalizer.transform(next_price),
        sum_insure_normalizer.transform(next_sum_insure),
    )

In [46]:
def compute_losses(
        model,
        items,
        ages_norm,
        prices_norm,
        sum_insures_norm,
        next_item,
        next_age_norm,
        next_price_norm,
        next_sum_insure_norm,
):
    items_logits, age_pred, price_pred, sum_insure_pred = model(
        [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
    )
    return (
        nn.functional.cross_entropy(items_logits, next_item),
        nn.functional.mse_loss(age_pred, next_age_norm),
        nn.functional.mse_loss(price_pred, next_price_norm),
        nn.functional.mse_loss(sum_insure_pred, next_sum_insure_norm),
        (items_logits, age_pred, price_pred, sum_insure_pred)
    )

In [47]:
def task_weights(loss_weigher, task_names=("item", "age", "price", "sum_insure")):
    weights = torch.exp(-loss_weigher.log_vars).detach()
    return {name: w.item() for name, w in zip(task_names, weights)}

In [48]:
def run_experiment(architecture, fusion_strategy, epochs=50, patience=3, check_every=5, lr=1e-3, verbose=True):
    tag = f"[{architecture}/{fusion_strategy}]"
    model, loss_weigher = build_model(architecture, fusion_strategy, num_items=num_items)
    optimizer = torch.optim.Adam(list(model.parameters()) + list(loss_weigher.parameters()), lr=lr)

    best_val_loss, best_state_dict, best_metrics = float("inf"), None, None
    check_without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()
        for items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure in next_item_train_loader:
            ages_norm, prices_norm, sum_insures_norm, next_age_norm, next_price_norm, next_sum_insure_norm = (
                normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure)
            )
            loss_item, loss_age, loss_price, loss_sum_insure, _ = compute_losses(
                model, items, ages_norm, prices_norm, sum_insures_norm,
                next_item, next_age_norm, next_price_norm, next_sum_insure_norm
            )
            loss = loss_weigher([loss_item, loss_age, loss_price, loss_sum_insure])

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        if (epoch+1) % check_every != 0:
            continue

        model.eval()
        correct, total = 0, 0
        age_err, price_err, sum_insure_err, val_loss_total = 0.0, 0.0, 0.0, 0.0

        with torch.no_grad():
            for items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure in next_item_val_loader:
                ages_norm, prices_norm, sum_insures_norm, next_age_norm, next_price_norm, next_sum_insure_norm = (
                    normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure)
                )
                loss_item, loss_age, loss_price, loss_sum_insure, (item_logits, age_pred, price_pred, sum_insure_pred) = (
                    compute_losses(
                        model, items, ages_norm, prices_norm, sum_insures_norm,
                        next_item, next_age_norm, next_price_norm, next_sum_insure_norm
                    )
                )
                val_loss_total += loss_weigher([loss_item, loss_age, loss_price, loss_sum_insure]).item()

                correct += ( item_logits.argmax(dim=-1) == next_item ).sum().item()
                total += next_item.numel()
                age_err += (age_normalizer.inverse_transform(age_pred) - next_age).abs().sum().item()
                price_err += (price_normalizer.inverse_transform(price_pred) - next_price).abs().sum().item()
                sum_insure_err += (sum_insure_normalizer.inverse_transform(sum_insure_pred) - next_sum_insure).abs().sum().item()

        n_val = len(next_item_val_loader.dataset)
        current_val_loss = val_loss_total / len(next_item_val_loader)
        metrics = {
            "epoch": epoch+1,
            "val_loss": current_val_loss,
            "item_acc": correct / total,
            "age_mae": age_err / n_val,
            "price_mae": price_err / n_val,
            "sum_insure_mae": sum_insure_err / n_val,
            "weights": task_weights(loss_weigher),
        }
        history.append(metrics)
        if verbose:
            w = metrics["weights"]
            print(
                f"{tag} epoch {epoch+1:3d} | val_loss {current_val_loss:.4f} | item_acc {metrics['item_acc']:.2%} "
                f"| age_mae {metrics['age_mae']:.2f} | price_mae {metrics['price_mae']:.2f} | sum_insure_mae {metrics['sum_insure_mae']:.2f}"
            )
            print(f"{tag}   weights: item={w['item']:.2f} age={w['age']:.2f} price={w['price']:.2f} sum_insure={w['sum_insure']:.2f}")

        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            best_state_dict = {k: v.clone() for k, v in model.state_dict().items()}
            best_metrics = metrics
            checks_without_improvement = 0
        
        else:
            checks_without_improvement += 1
            if checks_without_improvement >= patience:
                if verbose:
                    print(
                        f"{tag} no improvement for {patience} checks - stopping early at epoch {epoch+1}"
                    )
                break
    
    model.load_state_dict(best_state_dict)
    return {
        "architecture": architecture,
        "fusion": fusion_strategy,
        "model": model,
        "best_metrics": best_metrics,
        "history": history
    }

        

In [49]:
result = run_experiment(architecture="encoder", fusion_strategy="sum")
print(result["best_metrics"])

[encoder/sum] epoch   5 | val_loss 3.0712 | item_acc 15.94% | age_mae 1.26 | price_mae 1450.95 | sum_insure_mae 27982.74
[encoder/sum]   weights: item=0.82 age=1.25 price=0.90 sum_insure=0.94
[encoder/sum] epoch  10 | val_loss 2.9959 | item_acc 17.75% | age_mae 1.25 | price_mae 1413.31 | sum_insure_mae 30452.86
[encoder/sum]   weights: item=0.71 age=1.57 price=0.85 sum_insure=0.92
[encoder/sum] epoch  15 | val_loss 2.8365 | item_acc 17.03% | age_mae 1.27 | price_mae 1439.55 | sum_insure_mae 29124.18
[encoder/sum]   weights: item=0.63 age=1.96 price=0.82 sum_insure=0.93
[encoder/sum] epoch  20 | val_loss 2.6381 | item_acc 15.94% | age_mae 1.27 | price_mae 1482.19 | sum_insure_mae 28512.98
[encoder/sum]   weights: item=0.59 age=2.44 price=0.82 sum_insure=0.96
[encoder/sum] epoch  25 | val_loss 2.8266 | item_acc 13.77% | age_mae 1.27 | price_mae 1494.92 | sum_insure_mae 32478.13
[encoder/sum]   weights: item=0.57 age=3.04 price=0.86 sum_insure=1.01
[encoder/sum] epoch  30 | val_loss 2.761

In [50]:
result = run_experiment(architecture="encoder", fusion_strategy="concat_all")
print(result["best_metrics"])

[encoder/concat_all] epoch   5 | val_loss 3.0859 | item_acc 17.39% | age_mae 1.24 | price_mae 1378.68 | sum_insure_mae 28124.55
[encoder/concat_all]   weights: item=0.82 age=1.25 price=0.90 sum_insure=0.94
[encoder/concat_all] epoch  10 | val_loss 2.8586 | item_acc 17.75% | age_mae 1.24 | price_mae 1440.10 | sum_insure_mae 28477.85
[encoder/concat_all]   weights: item=0.71 age=1.57 price=0.84 sum_insure=0.92
[encoder/concat_all] epoch  15 | val_loss 2.7892 | item_acc 16.67% | age_mae 1.26 | price_mae 1520.56 | sum_insure_mae 30590.94
[encoder/concat_all]   weights: item=0.63 age=1.96 price=0.80 sum_insure=0.92
[encoder/concat_all] epoch  20 | val_loss 2.8458 | item_acc 15.22% | age_mae 1.26 | price_mae 1464.40 | sum_insure_mae 30650.64
[encoder/concat_all]   weights: item=0.59 age=2.44 price=0.78 sum_insure=0.94
[encoder/concat_all] epoch  25 | val_loss 2.7114 | item_acc 17.39% | age_mae 1.27 | price_mae 1524.50 | sum_insure_mae 29209.29
[encoder/concat_all]   weights: item=0.56 age=3.

In [51]:
result = run_experiment(architecture="encoder", fusion_strategy="concat_emb_summed")
print(result["best_metrics"])

[encoder/concat_emb_summed] epoch   5 | val_loss 3.0842 | item_acc 18.12% | age_mae 1.25 | price_mae 1409.39 | sum_insure_mae 29771.98
[encoder/concat_emb_summed]   weights: item=0.82 age=1.26 price=0.91 sum_insure=0.95
[encoder/concat_emb_summed] epoch  10 | val_loss 2.8327 | item_acc 16.67% | age_mae 1.26 | price_mae 1436.18 | sum_insure_mae 28625.62
[encoder/concat_emb_summed]   weights: item=0.71 age=1.57 price=0.85 sum_insure=0.92
[encoder/concat_emb_summed] epoch  15 | val_loss 2.7770 | item_acc 15.58% | age_mae 1.26 | price_mae 1377.54 | sum_insure_mae 29856.89
[encoder/concat_emb_summed]   weights: item=0.63 age=1.96 price=0.81 sum_insure=0.92
[encoder/concat_emb_summed] epoch  20 | val_loss 2.6406 | item_acc 19.57% | age_mae 1.27 | price_mae 1500.18 | sum_insure_mae 29111.97
[encoder/concat_emb_summed]   weights: item=0.59 age=2.45 price=0.79 sum_insure=0.93
[encoder/concat_emb_summed] epoch  25 | val_loss 2.7619 | item_acc 16.30% | age_mae 1.28 | price_mae 1490.59 | sum_insur

# Decoder Block

In [52]:
import torch
import torch.nn as nn
from package.gpt_decoder_block import GPTDecoderBlock

class DecoderBackbone(nn.Module):
    def __init__(self, fusion, d_model: int, n_heads: int, max_len: int, num_layers: int = 2):
        super().__init__()
        self.fusion = fusion
        self.layers = nn.ModuleList([
            GPTDecoderBlock(d_model, n_heads, max_len) for _ in range(num_layers)
        ])
    
    def forward(self, categorical_features, continuous_features, pad_mask_source):
        pad_mask = pad_mask_source == 0
        x = self.fusion(categorical_features, continuous_features)

        for layer in self.layers:
            x = layer(x, pad_mask)

        # return shape: (batch, seq_len, d_model)
        return x

In [53]:
class MultiTaskDecoderModel(nn.Module):
    def __init__(self, backbone, d_model: int, num_items: int):
        super().__init__()
        self.backbone = backbone
        self.item_head = nn.Linear(d_model, num_items + 1)
        self.age_head = nn.Linear(d_model, 1)
        self.price_head = nn.Linear(d_model, 1)
        self.sum_insure_head = nn.Linear(d_model, 1)
    
    def forward(self, categorical_features, continuous_features, pad_mask_source):
        x = self.backbone(categorical_features, continuous_features, pad_mask_source)
        ages_norm = continuous_features[0]
        delta_age = nn.functional.softplus(self.age_head(x))
        age_pred = ages_norm + delta_age

        return (
            self.item_head(x),
            age_pred,
            self.price_head(x),
            self.sum_insure_head(x)
        )

In [54]:
batch, seq_len, d_model, n_heads = 4, 8, 32, 4

fusion   = FusionEmbedding([num_items], num_continuous=3, d_model=d_model, strategy="sum")
backbone = DecoderBackbone(fusion, d_model=d_model, n_heads=n_heads, max_len=seq_len, num_layers=2)
model    = MultiTaskDecoderModel(backbone, d_model=d_model, num_items=num_items)

item_id    = torch.randint(0, num_items + 1, (batch, seq_len))
age        = torch.randn(batch, seq_len, 1)
price      = torch.randn(batch, seq_len, 1)
sum_insure = torch.randn(batch, seq_len, 1)

item_logits, age_pred, price_pred, sum_insure_pred = model(
    [item_id], [age, price, sum_insure], pad_mask_source=item_id
)

print(item_logits.shape)      # expect (4, 8, num_items+1)
print(age_pred.shape)         # expect (4, 8, 1)
print(price_pred.shape)       # expect (4, 8, 1)
print(sum_insure_pred.shape)  # expect (4, 8, 1)

torch.Size([4, 8, 9])
torch.Size([4, 8, 1])
torch.Size([4, 8, 1])
torch.Size([4, 8, 1])


In [55]:
def masked_mse(pred, target, mask):
    mask = mask.unsqueeze(-1)
    squared_error = (pred-target) ** 2 * mask
    return squared_error.sum() / mask.sum().clamp(min=1)

In [56]:
items, ages, prices, sum_insures = next(iter(train_loader))

ages_norm = age_normalizer.transform(ages)
prices_norm = price_normalizer.transform(prices)
sum_insures_norm = sum_insure_normalizer.transform(sum_insures)

item_logits, age_pred, price_pred, sum_insure_pred = model(
    [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
)

pred_item = item_logits[:, :-1]
pred_age = age_pred[:, :-1]
pred_price = price_pred[:, :-1]
pred_sum_insure = sum_insure_pred[:, :-1]

tgt_item = items[:, 1:]
tgt_age_norm = ages_norm[:, 1:]
tgt_price_norm = prices_norm[:, 1:]
tgt_sum_insure_norm = sum_insures_norm[:, 1:]

loss_item = nn.functional.cross_entropy(pred_item.transpose(1, 2), tgt_item, ignore_index=0)

valid_mask = tgt_item != 0

loss_age = masked_mse(pred_age, tgt_age_norm, valid_mask)
loss_price = masked_mse(pred_price, tgt_price_norm, valid_mask)
loss_sum_insure = masked_mse(pred_sum_insure, tgt_sum_insure_norm, valid_mask)

print(loss_item.item(), loss_age.item(), loss_price.item(), loss_sum_insure.item())

2.4070677757263184 0.11028852313756943 1.0962204933166504 1.1770578622817993


In [57]:
def encoder_batch_step(model, batch):
    items, ages, prices, sum_insures, next_item, next_age, next_price, next_sum_insure = batch
    ages_norm, prices_norm, sum_insures_norm, next_age_norm, next_price_norm, next_sum_insure_norm = (
        normalize_batch(ages, prices, sum_insures, next_age, next_price, next_sum_insure)
    )
    item_logits, age_pred, price_pred, sum_insure_pred = model(
        [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
    )

    losses = (
        nn.functional.cross_entropy(item_logits, next_item),
        nn.functional.mse_loss(age_pred, next_age_norm),
        nn.functional.mse_loss(price_pred, next_price_norm),
        nn.functional.mse_loss(sum_insure_pred, next_sum_insure_norm),
    )
    stats = {
        "correct": (item_logits.argmax(dim=-1)==next_item).sum().item(),
        "total": next_item.numel(),
        "age_err": (age_normalizer.inverse_transform(age_pred)-next_age).abs().sum().item(),
        "price_err": (price_normalizer.inverse_transform(price_pred)-next_price).abs().sum().item(),
        "sum_insure_err": (sum_insure_normalizer.inverse_transform(sum_insure_pred)-next_sum_insure).abs().sum().item(),
    }
    return losses, stats

In [58]:
def decoder_batch_step(model, batch):
    items, ages, prices, sum_insures = batch
    ages_norm = age_normalizer.transform(ages)
    prices_norm = price_normalizer.transform(prices)
    sum_insures_norm = sum_insure_normalizer.transform(sum_insures)

    item_logits, age_pred, price_pred, sum_insure_pred = model(
        [items], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items
    )

    pred_item, pred_age, pred_price, pred_sum_insure = (
        item_logits[:, :-1], age_pred[:, :-1], price_pred[:, :-1], sum_insure_pred[:, :-1]
    )
    tgt_item = items[:, 1:]
    tgt_age_norm, tgt_price_norm, tgt_sum_insure_norm = ages_norm[:, 1:], prices_norm[:, 1:], sum_insures_norm[:, 1:]
    valid_mask = tgt_item != 0

    losses = (
        nn.functional.cross_entropy(pred_item.transpose(1, 2), tgt_item, ignore_index=0),
        masked_mse(pred_age, tgt_age_norm, valid_mask),
        masked_mse(pred_price, tgt_price_norm, valid_mask),
        masked_mse(pred_sum_insure, tgt_sum_insure_norm, valid_mask),
    )

    tgt_age, tgt_price, tgt_sum_insure = ages[:, 1:], prices[:, 1:], sum_insures[:, 1:]
    mask3 = valid_mask.unsqueeze(-1)
    stats = {
        "correct": ((pred_item.argmax(dim=-1)==tgt_item) & valid_mask).sum().item(),
        "total": valid_mask.sum().item(),
        "age_err": (((age_normalizer.inverse_transform(pred_age)-tgt_age).abs())*mask3).sum().item(),
        "price_err": (((price_normalizer.inverse_transform(pred_price)-tgt_price).abs())*mask3).sum().item(),
        "sum_insure_err": (((sum_insure_normalizer.inverse_transform(pred_sum_insure)-tgt_sum_insure).abs())*mask3).sum().item(),
    }
    return losses, stats

In [59]:
def build_model(
        architecture,
        fusion_strategy,
        num_items,
        num_continuous=3,
        d_model=32,
        n_heads=4,
        num_layers=2,
        max_len=8,
):
    fusion = FusionEmbedding([num_items], num_continuous, d_model, strategy=fusion_strategy)
    if architecture == "encoder":
        backbone = EncoderCLSBackbone(fusion, d_model=d_model, n_heads=n_heads, max_len=max_len, num_layers=num_layers)
        model = MultiTaskModel(backbone, d_model=d_model, num_items=num_items)
    elif architecture == "decoder":
        backbone = DecoderBackbone(fusion, d_model=d_model, n_heads=n_heads, max_len=max_len, num_layers=num_layers)
        model = MultiTaskDecoderModel(backbone, d_model=d_model, num_items=num_items)
    else:
        raise ValueError(f"Unknown architecture: {architecture}")
    
    loss_weigher = UncertaintyWeightedLoss(task_types=["classification", "regression", "regression", "regression"])
    return model, loss_weigher

In [60]:
# this is considered to be a great way to setup parameters
BATCH_STEP = {"encoder": encoder_batch_step, "decoder": decoder_batch_step}
LOADERS = {
    "encoder": (next_item_train_loader, next_item_val_loader),
    "decoder": (train_loader, val_loader)
}

def run_experiment(
        architecture, 
        fusion_strategy,
        epochs=50,
        patience=3,
        check_every=5,
        lr=1e-3,
        verbose=True,
):
    tag = f"[{architecture}/{fusion_strategy}]"
    # num_items <- it should be passed as a param not injecting like this
    model, loss_weigher = build_model(architecture, fusion_strategy, num_items=num_items)
    optimizer = torch.optim.Adam(list(model.parameters())+list(loss_weigher.parameters()), lr=lr)

    batch_step = BATCH_STEP[architecture]
    train_loader_, val_loader_ = LOADERS[architecture]

    best_val_loss, best_state_dict, best_metrics = float("inf"), None, None
    checks_without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()
        for batch in train_loader_:
            losses, _ = batch_step(model, batch)
            loss = loss_weigher(list(losses))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if (epoch+1) % check_every != 0:
            continue

        model.eval()
        correct, total = 0, 0
        age_err, price_err, sum_insure_err, val_loss_total = 0.0, 0.0, 0.0, 0.0

        with torch.no_grad():
            for batch in val_loader_:
                losses, stats = batch_step(model, batch)
                val_loss_total += loss_weigher(list(losses)).item()
                correct += stats["correct"]
                total += stats["total"]
                age_err += stats["age_err"]
                price_err += stats["price_err"]
                sum_insure_err += stats["sum_insure_err"]

        current_val_loss = val_loss_total / len(val_loader_)
        metrics = {
            "epoch": epoch+1,
            "val_loss": current_val_loss,
            "item_acc": correct / total,
            "age_mae": age_err / total,
            "price_mae": price_err / total,
            "sum_insure_mae": sum_insure_err / total,
            "weights": task_weights(loss_weigher),
        }
        history.append(metrics)
        if verbose:
            w = metrics["weights"]
            print(
                f"{tag} epoch {epoch+1:3d} | val_loss {current_val_loss:.4f} | item_acc {metrics['item_acc']:.2%} "
                f"| age_mae {metrics['age_mae']:.2f} | price_mae {metrics['price_mae']:.2f} | sum_insure_mae {metrics['sum_insure_mae']:.2f}"
            )
            print(f"{tag}   weights: item={w['item']:.2f} age={w['age']:.2f} price={w['price']:.2f} sum_insure={w['sum_insure']:.2f}")

        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            best_state_dict = {k: v.clone() for k, v in model.state_dict().items()}
            best_metrics = metrics
            checks_without_improvement = 0
        else:
            checks_without_improvement += 1
            if checks_without_improvement >= patience:
                if verbose:
                    print(f"{tag} no improvement for {patience} checks - stopping early at epoch {epoch+1}")
                break
    
    model.load_state_dict(best_state_dict)
    return {
        "architecture": architecture,
        "fusion": fusion_strategy,
        "model": model,
        "best_metrics": best_metrics,
        "history": history
    }


In [61]:
result = run_experiment(architecture="decoder", fusion_strategy="sum")
print(result["best_metrics"])

[decoder/sum] epoch   5 | val_loss 3.2366 | item_acc 19.20% | age_mae 1.33 | price_mae 1409.14 | sum_insure_mae 27768.31
[decoder/sum]   weights: item=0.95 age=1.06 price=0.96 sum_insure=0.98
[decoder/sum] epoch  10 | val_loss 3.1787 | item_acc 16.67% | age_mae 1.30 | price_mae 1374.49 | sum_insure_mae 28263.42
[decoder/sum]   weights: item=0.91 age=1.12 price=0.93 sum_insure=0.97
[decoder/sum] epoch  15 | val_loss 3.1788 | item_acc 17.03% | age_mae 1.29 | price_mae 1409.66 | sum_insure_mae 28905.22
[decoder/sum]   weights: item=0.87 age=1.18 price=0.91 sum_insure=0.96
[decoder/sum] epoch  20 | val_loss 3.2383 | item_acc 17.03% | age_mae 1.29 | price_mae 1510.43 | sum_insure_mae 29846.97
[decoder/sum]   weights: item=0.83 age=1.25 price=0.90 sum_insure=0.97
[decoder/sum] epoch  25 | val_loss 3.2663 | item_acc 18.84% | age_mae 1.30 | price_mae 1549.13 | sum_insure_mae 29696.95
[decoder/sum]   weights: item=0.80 age=1.32 price=0.89 sum_insure=0.98
[decoder/sum] no improvement for 3 check

# Experiments

In [62]:
import itertools
import pandas as pd
torch.manual_seed(55)

architectures = [
    "encoder", 
    "decoder"
]
fusion_strategies = ["sum", "concat_all", "concat_emb_summed"]

all_results = {}
for architecture, fusion_strategy in itertools.product(architectures, fusion_strategies):
    print(f"\n{'='*70}\n{architecture}+{fusion_strategy}\n{'='*70}")
    all_results[(architecture, fusion_strategy)] = run_experiment(architecture, fusion_strategy)


encoder+sum
[encoder/sum] epoch   5 | val_loss 3.0908 | item_acc 19.93% | age_mae 1.28 | price_mae 1366.38 | sum_insure_mae 29218.74
[encoder/sum]   weights: item=0.82 age=1.25 price=0.91 sum_insure=0.95
[encoder/sum] epoch  10 | val_loss 2.9340 | item_acc 19.20% | age_mae 1.28 | price_mae 1398.14 | sum_insure_mae 28537.29
[encoder/sum]   weights: item=0.71 age=1.57 price=0.85 sum_insure=0.94
[encoder/sum] epoch  15 | val_loss 2.9337 | item_acc 17.03% | age_mae 1.28 | price_mae 1420.70 | sum_insure_mae 30582.68
[encoder/sum]   weights: item=0.64 age=1.96 price=0.82 sum_insure=0.95
[encoder/sum] epoch  20 | val_loss 2.8410 | item_acc 15.94% | age_mae 1.28 | price_mae 1484.06 | sum_insure_mae 29340.11
[encoder/sum]   weights: item=0.59 age=2.44 price=0.81 sum_insure=1.00
[encoder/sum] epoch  25 | val_loss 2.9560 | item_acc 17.03% | age_mae 1.28 | price_mae 1607.63 | sum_insure_mae 30434.75
[encoder/sum]   weights: item=0.57 age=3.04 price=0.84 sum_insure=1.07
[encoder/sum] epoch  30 | v

In [63]:
rows = []
for (architecture, fusion_strategy), result in all_results.items():
    m = result["best_metrics"]
    rows.append({
        "architecture": architecture,
        "fusion": fusion_strategy,
        "best_epoch": m["epoch"],
        "val_loss": round(m["val_loss"], 4),
        "item_acc": round(m["item_acc"], 4),
        "age_mae": round(m["age_mae"], 2),
        "price_mae": round(m["price_mae"], 2),
        "sum_insure_mae": round(m["sum_insure_mae"], 2),
    })

In [64]:
summary = pd.DataFrame(rows).sort_values(["architecture", "fusion"]).reset_index(drop=True)
summary.to_csv("./runs/experiment_summary.csv", index=False)
summary.sort_values(by=['item_acc'], ascending=[False])

,architecture,fusion,best_epoch,val_loss,item_acc,age_mae,price_mae,sum_insure_mae
3,encoder,concat_all,40,2.4915,0.2065,1.30,1482.46,28263.83
1,decoder,concat_emb_summed,15,3.0846,0.1884,1.27,1453.01,28693.65
2,decoder,sum,15,3.1075,0.1848,1.26,1487.04,28327.47
0,decoder,concat_all,15,3.0917,0.1739,1.26,1457.45,28979.08
5,encoder,sum,20,2.8410,0.1594,1.28,1484.06,29340.11
4,encoder,concat_emb_summed,20,2.6633,0.1558,1.26,1502.26,28227.96


In [65]:
import pickle
with open("./runs/experiment_histories.pkl", "wb") as f:
    pickle.dump({k:v["history"] for k, v in all_results.items()}, f)

In [68]:
import os
os.makedirs("./runs/models", exist_ok=True)

for (architecture, fusion_strategy), result in all_results.items():
    state = {
        "architecture": architecture,
        "fusion_strategy": fusion_strategy,
        "num_items": num_items,
        "state_dict": result["model"].state_dict(),
        "best_metrics": result["best_metrics"],
    }
    path = f"./runs/models/{architecture}_{fusion_strategy}.pt"
    torch.save(state, path)
    print(f"saved {path}")


saved ./runs/models/encoder_sum.pt
saved ./runs/models/encoder_concat_all.pt
saved ./runs/models/encoder_concat_emb_summed.pt
saved ./runs/models/decoder_sum.pt
saved ./runs/models/decoder_concat_all.pt
saved ./runs/models/decoder_concat_emb_summed.pt


In [69]:
def load_model(architecture, fusion_strategy, path_dir="./runs/models"):
    path = f"{path_dir}/{architecture}_{fusion_strategy}.pt"
    checkpoint = torch.load(path, weights_only=False)
    model, _ = build_model(
        checkpoint["architecture"],
        checkpoint["fusion_strategy"],
        num_items=checkpoint["num_items"],
    )
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model, checkpoint["best_metrics"]

In [70]:
enc_model, enc_metrics = load_model("encoder", "concat_all")
dec_model, dec_metrics = load_model("decoder", "concat_all")
print(enc_metrics)
print(dec_metrics)

{'epoch': 40, 'val_loss': 2.4915405909220376, 'item_acc': 0.20652173913043478, 'age_mae': 1.300182632778002, 'price_mae': 1482.462055593297, 'sum_insure_mae': 28263.8272192029, 'weights': {'item': 0.5618603229522705, 'age': 5.736858367919922, 'price': 0.9496573209762573, 'sum_insure': 1.199428677558899}}
{'epoch': 15, 'val_loss': 3.0917479197184243, 'item_acc': 0.17391304347826086, 'age_mae': 1.2559097953464673, 'price_mae': 1457.453804347826, 'sum_insure_mae': 28979.077445652172, 'weights': {'item': 0.8635213375091553, 'age': 1.1806689500808716, 'price': 0.9023560285568237, 'sum_insure': 0.9526001811027527}}


# Probe

# Leave one event out

In [71]:
test_indices = test_set.indices
sample_idx = next(
    i for i in test_indices if len(dataset.samples[i][0]) >= 6
)
items_full, ages_full, prices_full, sum_insures_full = dataset.samples[sample_idx]
print(f"customer idx {sample_idx}, n_events={len(items_full)}")
print("items:", items_full)
print("ages:", ages_full)

# prefix = everything except the last event; the last event is the ground-truth "next"
items_prefix, ages_prefix = items_full[:-1], ages_full[:-1]
prices_prefix, sum_insures_prefix = prices_full[:-1], sum_insures_full[:-1]
true_next_item = items_full[-1]
true_next_age, true_next_price, true_next_sum_insure = ages_full[-1], prices_full[-1], sum_insures_full[-1]


customer idx 438, n_events=8
items: [2, 6, 6, 6, 6, 6, 3, 7]
ages: [38.0, 42.0, 45.0, 47.0, 50.0, 55.0, 59.0, 62.0]


In [80]:
def predict_one(model, architecture, items_list, ages_list, prices_list, sum_insures_list):
    length = len(items_list)
    items_t = torch.tensor(pad(items_list, MAX_EVENTS), dtype=torch.long).unsqueeze(0)
    ages_t = torch.tensor(pad(ages_list, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(0).unsqueeze(-1)
    prices_t = torch.tensor(pad(prices_list, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(0).unsqueeze(-1)
    sum_insures_t = torch.tensor(pad(sum_insures_list, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(0).unsqueeze(-1)

    ages_norm = age_normalizer.transform(ages_t)
    prices_norm = price_normalizer.transform(prices_t)
    sum_insures_norm = sum_insure_normalizer.transform(sum_insures_t)

    with torch.no_grad():
        item_logits, age_pred, price_pred, sum_insure_pred = model(
            [items_t], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items_t
        )

    if architecture == "decoder":
        item_logits = item_logits[:, length - 1]
        age_pred = age_pred[:, length - 1]
        price_pred = price_pred[:, length - 1]
        sum_insure_pred = sum_insure_pred[:, length - 1]

    item_probs = item_logits.softmax(dim=-1).squeeze(0)
    return {
        "item_probs": item_probs,
        "item_pred": item_probs.argmax().item(),
        "age": age_normalizer.inverse_transform(age_pred).item(),
        "price": price_normalizer.inverse_transform(price_pred).item(),
        "sum_insure": sum_insure_normalizer.inverse_transform(sum_insure_pred).item(),
    }


In [81]:
baseline_enc = predict_one(enc_model, "encoder", items_prefix, ages_prefix, prices_prefix, sum_insures_prefix)
baseline_dec = predict_one(dec_model, "decoder", items_prefix, ages_prefix, prices_prefix, sum_insures_prefix)

print("true next item:", true_next_item, " true next age/price/sum_insure:", true_next_age, true_next_price, true_next_sum_insure)
print("encoder baseline:", baseline_enc)
print("decoder baseline:", baseline_dec)


true next item: 7  true next age/price/sum_insure: 62.0 2242.1504799334803 80395.67726553287
encoder baseline: {'item_probs': tensor([0.0005, 0.0322, 0.1239, 0.2260, 0.3989, 0.0144, 0.0153, 0.1378, 0.0510]), 'item_pred': 4, 'age': 62.335025787353516, 'price': 1192.016845703125, 'sum_insure': 130152.875}
decoder baseline: {'item_probs': tensor([0.0031, 0.0403, 0.0331, 0.1880, 0.2648, 0.1498, 0.0933, 0.1917, 0.0359]), 'item_pred': 4, 'age': 61.85383605957031, 'price': 2486.207275390625, 'sum_insure': 96927.578125}


In [77]:
def run_loeo(model, architecture, items_list, ages_list, prices_list, sum_insures_list, baseline):
    length = len(items_list)
    # decoder can't occlude its own read-out position (last real event) without
    # corrupting the query doing the reading, not just a memory being read — encoder's
    # CLS is a separate token, so it has no such restriction
    positions = range(length) if architecture == "encoder" else range(1, length - 1)

    rows = []
    for i in positions:
        items_occluded = items_list.copy()
        items_occluded[i] = 0
        occluded = predict_one(model, architecture, items_occluded, ages_list, prices_list, sum_insures_list)
        rows.append({
            "position": i,
            "occluded_item": items_list[i],
            "item_prob_true_delta": baseline["item_prob_true"] - occluded["item_prob_true"],
            "age_delta": baseline["age"] - occluded["age"],
            "price_delta": baseline["price"] - occluded["price"],
            "sum_insure_delta": baseline["sum_insure"] - occluded["sum_insure"],
        })
    return pd.DataFrame(rows)


In [78]:
loeo_enc = run_loeo(enc_model, "encoder", items_prefix, ages_prefix, prices_prefix, sum_insures_prefix, baseline_enc)
loeo_dec = run_loeo(dec_model, "decoder", items_prefix, ages_prefix, prices_prefix, sum_insures_prefix, baseline_dec)

print("Encoder LOEO (all prefix positions):")
print(loeo_enc.sort_values("item_prob_true_delta", key=abs, ascending=False).reset_index(drop=True))

print("\nDecoder LOEO (excludes last position, see note above):")
print(loeo_dec.sort_values("item_prob_true_delta", key=abs, ascending=False).reset_index(drop=True))


Encoder LOEO (all prefix positions):
   position  occluded_item  item_prob_true_delta  age_delta  price_delta  \
0         0              2              0.074859   4.277447   227.390381   
1         6              3             -0.072299   3.997475  -383.770020   
2         4              6              0.038740   4.098293   216.113220   
3         5              6              0.024405   3.982002   179.489746   
4         3              6              0.019923   3.917664   128.607422   
5         1              6              0.006616   4.002968    26.415649   
6         2              6             -0.003301   4.062309   -10.210083   

   sum_insure_delta  
0     -19479.156250  
1      25822.921875  
2     -13532.421875  
3        856.328125  
4      -3572.578125  
5       1860.515625  
6       5255.937500  

Decoder LOEO (excludes last position, see note above):
   position  occluded_item  item_prob_true_delta  age_delta  price_delta  \
0         1              6              0.0041

In [79]:
loeo_dec

,position,occluded_item,item_prob_true_delta,age_delta,price_delta,sum_insure_delta
0,1,6,0.004180,-0.013840,0.748535,90.859375
1,2,6,-0.001216,-0.005932,-60.920166,-1692.265625
2,3,6,0.002411,0.006874,-11.670898,851.492188
3,4,6,-0.002514,0.011658,-76.635742,-733.515625
4,5,6,-0.001941,-0.001373,-49.810303,-768.000000


In [82]:
def collect_test_customers(min_len=6, n=20):
    customers = []
    for i in test_set.indices:
        items_full, ages_full, prices_full, sum_insures_full = dataset.samples[i]
        if len(items_full) >= min_len:
            customers.append((i, items_full, ages_full, prices_full, sum_insures_full))
        if len(customers) >= n:
            break
    return customers

customers = collect_test_customers(min_len=6, n=20)
print(f"collected {len(customers)} customers, lengths: {[len(c[1]) for c in customers]}")


collected 20 customers, lengths: [8, 8, 7, 6, 7, 8, 6, 6, 6, 7, 8, 6, 8, 8, 8, 6, 6, 7, 8, 7]


In [84]:
def loeo_for_customer(model, architecture, items_full, ages_full, prices_full, sum_insures_full, customer_idx):
    items_prefix, ages_prefix = items_full[:-1], ages_full[:-1]
    prices_prefix, sum_insures_prefix = prices_full[:-1], sum_insures_full[:-1]
    true_next_item = items_full[-1]
    length = len(items_prefix)

    baseline = predict_one(model, architecture, items_prefix, ages_prefix, prices_prefix, sum_insures_prefix)
    baseline_prob_true = baseline["item_probs"][true_next_item].item()

    positions = range(length) if architecture == "encoder" else range(1, length - 1)

    rows = []
    for i in positions:
        items_occluded = items_prefix.copy()
        items_occluded[i] = 0
        occluded = predict_one(model, architecture, items_occluded, ages_prefix, prices_prefix, sum_insures_prefix)
        occluded_prob_true = occluded["item_probs"][true_next_item].item()
        rows.append({
            "customer_idx": customer_idx,
            "distance_from_end": (length - 1) - i,
            "occluded_item": items_prefix[i],
            "item_prob_true_delta": baseline_prob_true - occluded_prob_true,
            "age_delta": baseline["age"] - occluded["age"],
            "price_delta": baseline["price"] - occluded["price"],
            "sum_insure_delta": baseline["sum_insure"] - occluded["sum_insure"],
        })
    return pd.DataFrame(rows)

def run_loeo_many(model, architecture, customers):
    return pd.concat(
        [
            loeo_for_customer(model, architecture, items_full, ages_full, prices_full, sum_insures_full, customer_idx)
            for customer_idx, items_full, ages_full, prices_full, sum_insures_full in customers
        ],
        ignore_index=True,
    )


loeo_enc_all = run_loeo_many(enc_model, "encoder", customers)
loeo_dec_all = run_loeo_many(dec_model, "decoder", customers)


In [85]:
loeo_enc_all

,customer_idx,distance_from_end,occluded_item,item_prob_true_delta,age_delta,price_delta,sum_insure_delta
0,438,6,2,0.074859,4.277447,227.390381,-19479.156250
1,438,5,6,0.006616,4.002968,26.415649,1860.515625
2,438,4,6,-0.003301,4.062309,-10.210083,5255.937500
3,438,3,6,0.019923,3.917664,128.607422,-3572.578125
4,438,2,6,0.038740,4.098293,216.113220,-13532.421875
...,...,...,...,...,...,...,...
116,170,4,3,-0.009608,3.176640,214.921753,-2104.558594
117,170,3,8,0.003080,3.016140,-18.435059,-2908.619141
118,170,2,5,-0.005826,3.114574,22.374023,-838.812500
119,170,1,1,-0.008587,3.181404,292.422363,682.076172


In [86]:
loeo_dec_all

,customer_idx,distance_from_end,occluded_item,item_prob_true_delta,age_delta,price_delta,sum_insure_delta
0,438,5,6,0.004180,-0.013840,0.748535,90.859375
1,438,4,6,-0.001216,-0.005932,-60.920166,-1692.265625
2,438,3,6,0.002411,0.006874,-11.670898,851.492188
3,438,2,6,-0.002514,0.011658,-76.635742,-733.515625
4,438,1,6,-0.001941,-0.001373,-49.810303,-768.000000
...,...,...,...,...,...,...,...
76,365,1,3,-0.007160,0.035866,-12.876831,-947.152344
77,170,4,3,0.003081,-0.011459,-18.375122,-86.175781
78,170,3,8,0.000029,0.083054,8.572510,2370.027344
79,170,2,5,-0.008005,-0.043228,23.596436,-2030.492188


In [87]:
def summarize_by_distance(loeo_all):
    return (
        loeo_all
        .assign(
            abs_item=loeo_all["item_prob_true_delta"].abs(),
            abs_price=loeo_all["price_delta"].abs(),
            abs_sum_insure=loeo_all["sum_insure_delta"].abs(),
        )
        .groupby("distance_from_end")
        .agg(
            n=("customer_idx", "count"),
            item_delta_mean=("item_prob_true_delta", "mean"),
            abs_item_mean=("abs_item", "mean"),
            price_delta_mean=("price_delta", "mean"),
            abs_price_mean=("abs_price", "mean"),
            sum_insure_delta_mean=("sum_insure_delta", "mean"),
            abs_sum_insure_mean=("abs_sum_insure", "mean"),
        )
        .reset_index()
    )

print("Encoder — impact by distance from end of prefix:")
print(summarize_by_distance(loeo_enc_all))
print("\nDecoder — impact by distance from end of prefix:")
print(summarize_by_distance(loeo_dec_all))


Encoder — impact by distance from end of prefix:
   distance_from_end   n  item_delta_mean  abs_item_mean  price_delta_mean  \
0                  0  20         0.002959       0.032120         47.830676   
1                  1  20         0.011169       0.027792         51.606009   
2                  2  20         0.001846       0.018795        141.729364   
3                  3  20         0.002915       0.044419       -157.692316   
4                  4  20        -0.012825       0.030457        274.133368   
5                  5  13         0.005475       0.017663        -92.008878   
6                  6   8        -0.005632       0.044454        142.305725   

   abs_price_mean  sum_insure_delta_mean  abs_sum_insure_mean  
0      292.458765            1979.043945          5845.785547  
1      274.805899            -266.454687          4549.623633  
2      247.821881           -2119.058691          3521.976855  
3      481.016187            3587.168359          6891.421680  
4     

In [88]:
summarize_by_distance(loeo_enc_all)

,distance_from_end,n,item_delta_mean,abs_item_mean,price_delta_mean,abs_price_mean,sum_insure_delta_mean,abs_sum_insure_mean
0,0,20,0.002959,0.032120,47.830676,292.458765,1979.043945,5845.785547
1,1,20,0.011169,0.027792,51.606009,274.805899,-266.454687,4549.623633
2,2,20,0.001846,0.018795,141.729364,247.821881,-2119.058691,3521.976855
3,3,20,0.002915,0.044419,-157.692316,481.016187,3587.168359,6891.421680
4,4,20,-0.012825,0.030457,274.133368,380.566107,-5144.858887,7779.519043
5,5,13,0.005475,0.017663,-92.008878,195.847079,-196.047175,4999.866286
6,6,8,-0.005632,0.044454,142.305725,358.714111,-6110.082275,6110.082275


In [89]:
summarize_by_distance(loeo_dec_all)

,distance_from_end,n,item_delta_mean,abs_item_mean,price_delta_mean,abs_price_mean,sum_insure_delta_mean,abs_sum_insure_mean
0,1,20,-0.000630,0.002615,-37.079538,44.627103,-790.807910,1996.327051
1,2,20,-0.000640,0.003230,7.549762,41.504474,-20.300488,1310.190332
2,3,20,0.000861,0.003197,6.854431,27.195734,945.211328,1817.637305
3,4,13,0.000743,0.002686,2.529086,29.746521,-36.073317,895.892428
4,5,8,0.000217,0.002284,-10.370728,29.524078,-951.491211,1190.934570


# Linear probes

In [90]:
def collect_all_test_customers(min_len=2):
    customers = []
    for i in test_set.indices:
        items_full, ages_full, prices_full, sum_insures_full = dataset.samples[i]
        if len(items_full) >= min_len:
            customers.append((i, items_full, ages_full, prices_full, sum_insures_full))
    return customers

probe_customers = collect_all_test_customers(min_len=2)
print(f"{len(probe_customers)} test customers available for probing")


75 test customers available for probing


In [91]:
def extract_representation(model, architecture, items_list, ages_list, prices_list, sum_insures_list):
    length = len(items_list)
    items_t = torch.tensor(pad(items_list, MAX_EVENTS), dtype=torch.long).unsqueeze(0)
    ages_t = torch.tensor(pad(ages_list, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(0).unsqueeze(-1)
    prices_t = torch.tensor(pad(prices_list, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(0).unsqueeze(-1)
    sum_insures_t = torch.tensor(pad(sum_insures_list, MAX_EVENTS, 0.0), dtype=torch.float).unsqueeze(0).unsqueeze(-1)

    ages_norm = age_normalizer.transform(ages_t)
    prices_norm = price_normalizer.transform(prices_t)
    sum_insures_norm = sum_insure_normalizer.transform(sum_insures_t)

    with torch.no_grad():
        rep = model.backbone([items_t], [ages_norm, prices_norm, sum_insures_norm], pad_mask_source=items_t)

    if architecture == "decoder":
        rep = rep[:, length - 1]  # same read-out position as everywhere else in this notebook

    return rep.squeeze(0).numpy()


In [92]:
import numpy as np

def build_probe_dataset(model, architecture, customers):
    X, y_item, y_age, y_price, y_sum_insure = [], [], [], [], []
    for customer_idx, items_full, ages_full, prices_full, sum_insures_full in customers:
        items_prefix, ages_prefix = items_full[:-1], ages_full[:-1]
        prices_prefix, sum_insures_prefix = prices_full[:-1], sum_insures_full[:-1]
        rep = extract_representation(model, architecture, items_prefix, ages_prefix, prices_prefix, sum_insures_prefix)
        X.append(rep)
        y_item.append(items_full[-1])
        y_age.append(ages_full[-1])
        y_price.append(prices_full[-1])
        y_sum_insure.append(sum_insures_full[-1])
    return np.array(X), np.array(y_item), np.array(y_age), np.array(y_price), np.array(y_sum_insure)


In [93]:
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score

def run_linear_probes(model, architecture, customers, seed=55):
    X, y_item, y_age, y_price, y_sum_insure = build_probe_dataset(model, architecture, customers)
    (X_train, X_test, item_train, item_test, age_train, age_test,
     price_train, price_test, si_train, si_test) = train_test_split(
        X, y_item, y_age, y_price, y_sum_insure, test_size=0.3, random_state=seed
    )

    item_probe = LogisticRegression(max_iter=1000).fit(X_train, item_train)
    age_probe = Ridge().fit(X_train, age_train)
    price_probe = Ridge().fit(X_train, price_train)
    si_probe = Ridge().fit(X_train, si_train)

    return {
        "item_acc": accuracy_score(item_test, item_probe.predict(X_test)),
        "age_r2": r2_score(age_test, age_probe.predict(X_test)),
        "price_r2": r2_score(price_test, price_probe.predict(X_test)),
        "sum_insure_r2": r2_score(si_test, si_probe.predict(X_test)),
        "n_train": len(X_train), "n_test": len(X_test),
    }

probe_results = {}
for architecture in ["encoder", "decoder"]:
    for fusion_strategy in ["sum", "concat_all", "concat_emb_summed"]:
        model, _ = load_model(architecture, fusion_strategy)
        probe_results[(architecture, fusion_strategy)] = run_linear_probes(model, architecture, probe_customers)

probe_df = pd.DataFrame([
    {"architecture": a, "fusion": f, **r} for (a, f), r in probe_results.items()
]).sort_values(["architecture", "fusion"])
print(probe_df)


  architecture             fusion  item_acc    age_r2  price_r2  \
4      decoder         concat_all  0.130435  0.954656 -1.145629   
5      decoder  concat_emb_summed  0.173913  0.934533 -0.944945   
3      decoder                sum  0.086957  0.933947 -0.068125   
1      encoder         concat_all  0.130435  0.704876  0.180471   
2      encoder  concat_emb_summed  0.086957  0.794300 -0.066364   
0      encoder                sum  0.173913  0.835082 -1.041718   

   sum_insure_r2  n_train  n_test  
4       0.319891       52      23  
5       0.201948       52      23  
3       0.040429       52      23  
1       0.096400       52      23  
2       0.305340       52      23  
0       0.182986       52      23  


In [94]:
probe_df

,architecture,fusion,item_acc,age_r2,price_r2,sum_insure_r2,n_train,n_test
4,decoder,concat_all,0.130435,0.954656,-1.145629,0.319891,52,23
5,decoder,concat_emb_summed,0.173913,0.934533,-0.944945,0.201948,52,23
3,decoder,sum,0.086957,0.933947,-0.068125,0.040429,52,23
1,encoder,concat_all,0.130435,0.704876,0.180471,0.096400,52,23
2,encoder,concat_emb_summed,0.086957,0.794300,-0.066364,0.305340,52,23
0,encoder,sum,0.173913,0.835082,-1.041718,0.182986,52,23


# Visualize

In [97]:
import umap
from sklearn.decomposition import PCA

def get_item_embeddings(model):
    return model.backbone.fusion.categorical_embs[0].weight.detach().numpy()  # (num_items+1, d_model)

item_emb = get_item_embeddings(enc_model)
item_ids = list(range(1, num_items + 1))  # skip index 0 (padding)
item_vectors = item_emb[item_ids]
item_names = [CATALOG[i][0] for i in item_ids]
item_categories = [CATALOG[i][1] for i in item_ids]

# only 8 points -- UMAP's neighbor graph isn't very meaningful at this scale, so keep
# n_neighbors small and treat this as illustrative, not statistically rigorous
reducer = umap.UMAP(n_neighbors=4, random_state=55)
item_umap = reducer.fit_transform(item_vectors)

item_df = pd.DataFrame({
    "item_id": item_ids, "name": item_names, "category": item_categories,
    "umap_x": item_umap[:, 0], "umap_y": item_umap[:, 1],
})
print(item_df)


d:\a-study-on-transformer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\a-study-on-transformer\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


   item_id                     name         category     umap_x     umap_y
0        1             SmartTerm 20             Term -14.605187 -10.479037
1        2          Term100 Protect             Term -12.650943 -10.197385
2        3    LegacyGold Whole Life        WholeLife -13.392186 -10.062581
3        4      Prestige Whole Life        WholeLife -12.828670  -9.588595
4        5  WealthBuilder Endowment        Endowment -14.232997  -9.909460
5        6    SavingsPlan Endowment        Endowment -14.756630 -11.741305
6        7       FlexiInvest Linked              ILP -13.727484 -11.345747
7        8                CI Shield  CriticalIllness -14.193013 -12.093392


In [99]:
item_df

,item_id,name,category,umap_x,umap_y
0,1,SmartTerm 20,Term,-14.605187,-10.479037
1,2,Term100 Protect,Term,-12.650943,-10.197385
2,3,LegacyGold Whole Life,WholeLife,-13.392186,-10.062581
3,4,Prestige Whole Life,WholeLife,-12.828670,-9.588595
4,5,WealthBuilder Endowment,Endowment,-14.232997,-9.909460
5,6,SavingsPlan Endowment,Endowment,-14.756630,-11.741305
6,7,FlexiInvest Linked,ILP,-13.727484,-11.345747
7,8,CI Shield,CriticalIllness,-14.193013,-12.093392


In [98]:
def sweep_continuous_projection(model, proj_index, values_norm):
    proj = model.backbone.fusion.continuous_projs[proj_index]
    x = torch.tensor(values_norm, dtype=torch.float).unsqueeze(-1)
    with torch.no_grad():
        return proj(x).numpy()

sweep_norm = np.linspace(-3, 3, 100)  # normalized units, ~full realistic range

age_vectors = sweep_continuous_projection(enc_model, 0, sweep_norm)
price_vectors = sweep_continuous_projection(enc_model, 1, sweep_norm)
sum_insure_vectors = sweep_continuous_projection(enc_model, 2, sweep_norm)

age_real = age_normalizer.inverse_transform(torch.tensor(sweep_norm)).numpy()
price_real = price_normalizer.inverse_transform(torch.tensor(sweep_norm)).numpy()
sum_insure_real = sum_insure_normalizer.inverse_transform(torch.tensor(sweep_norm)).numpy()

for name, vectors in [("age", age_vectors), ("price", price_vectors), ("sum_insure", sum_insure_vectors)]:
    pca = PCA(n_components=2).fit(vectors)
    print(f"{name}: top-component variance explained = {pca.explained_variance_ratio_[0]:.4f}")

age_umap = umap.UMAP(random_state=55).fit_transform(age_vectors)
price_umap = umap.UMAP(random_state=55).fit_transform(price_vectors)
sum_insure_umap = umap.UMAP(random_state=55).fit_transform(sum_insure_vectors)


age: top-component variance explained = 1.0000
price: top-component variance explained = 1.0000
sum_insure: top-component variance explained = 1.0000


d:\a-study-on-transformer\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\a-study-on-transformer\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\a-study-on-transformer\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [101]:
import plotly.express as px

fig = px.scatter(
    item_df, x="umap_x", y="umap_y", color="category", text="name",
    title="Item embedding table (UMAP) — colored by insurance category",
)
fig.update_traces(textposition="top center", marker=dict(size=14))
fig.update_layout(width=700, height=500)
fig.show()


In [102]:
age_df = pd.DataFrame({"umap_x": age_umap[:, 0], "umap_y": age_umap[:, 1], "age": age_real})
price_df = pd.DataFrame({"umap_x": price_umap[:, 0], "umap_y": price_umap[:, 1], "price": price_real})
sum_insure_df = pd.DataFrame({"umap_x": sum_insure_umap[:, 0], "umap_y": sum_insure_umap[:, 1], "sum_insure": sum_insure_real})

px.scatter(age_df, x="umap_x", y="umap_y", color="age", color_continuous_scale="Viridis",
           title="Age projection (UMAP) — colored by real age").show()

px.scatter(price_df, x="umap_x", y="umap_y", color="price", color_continuous_scale="Viridis",
           title="Price projection (UMAP) — colored by real price").show()

px.scatter(sum_insure_df, x="umap_x", y="umap_y", color="sum_insure", color_continuous_scale="Viridis",
           title="Sum insured projection (UMAP) — colored by real sum insured").show()


In [103]:
probe_customers = collect_all_test_customers(min_len=2)  # recompute in case kernel state moved on

X_customers, y_item, y_age, y_price, y_sum_insure = build_probe_dataset(enc_model, "encoder", probe_customers)

reducer = umap.UMAP(random_state=55)
customer_umap = reducer.fit_transform(X_customers)

customer_df = pd.DataFrame({
    "umap_x": customer_umap[:, 0],
    "umap_y": customer_umap[:, 1],
    "item": y_item.astype(str),   # categorical, so plotly treats it as discrete colors
    "age": y_age,
    "price": y_price,
    "sum_insure": y_sum_insure,
})


d:\a-study-on-transformer\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [104]:
px.scatter(customer_df, x="umap_x", y="umap_y", color="item",
           title="Customer representation (UMAP) — colored by true next item").show()

px.scatter(customer_df, x="umap_x", y="umap_y", color="age", color_continuous_scale="Viridis",
           title="Customer representation (UMAP) — colored by true next age").show()

px.scatter(customer_df, x="umap_x", y="umap_y", color="price", color_continuous_scale="Viridis",
           title="Customer representation (UMAP) — colored by true next price").show()

px.scatter(customer_df, x="umap_x", y="umap_y", color="sum_insure", color_continuous_scale="Viridis",
           title="Customer representation (UMAP) — colored by true next sum insured").show()


In [105]:
customer_df["log_price"] = np.log1p(customer_df["price"])
customer_df["log_sum_insure"] = np.log1p(customer_df["sum_insure"])

px.scatter(customer_df, x="umap_x", y="umap_y", color="log_price", color_continuous_scale="Viridis",
           title="Customer representation (UMAP) — colored by log(true next price)").show()

px.scatter(customer_df, x="umap_x", y="umap_y", color="log_sum_insure", color_continuous_scale="Viridis",
           title="Customer representation (UMAP) — colored by log(true next sum insured)").show()


# Appendices

In [66]:
import torch
from package.gpt_decoder_block import GPTDecoderBlock

torch.manual_seed(0)

d_model, n_heads, max_len = 16, 4, 8
decoder = GPTDecoderBlock(d_model, n_heads, max_len)
decoder.eval()   # no dropout in this block, but good habit regardless

# one sequence of 8 "events" (a..h) — random vectors standing in for whatever
# FusionEmbedding would have produced. The specific values don't matter — what
# matters is that the SAME decoder weights process them both ways below.
full_sequence = torch.randn(1, 8, d_model)   # (batch=1, seq_len=8, d_model)

with torch.no_grad():
    full_out = decoder(full_sequence)   # ONE forward pass over all 8 positions at once

print(f"{'prefix':<10}{'last-pos idx':<14}{'identical?':<12}sample values (first 3 dims)")
for prefix_len in range(1, 9):
    prefix = full_sequence[:, :prefix_len]        # e.g. prefix_len=3 -> just [a, b, c]
    with torch.no_grad():
        prefix_out = decoder(prefix)               # a SEPARATE forward pass, just this prefix

    from_full   = full_out[:, prefix_len - 1]       # position (prefix_len-1), read from the FULL pass
    from_prefix = prefix_out[:, -1]                 # the LAST position of the truncated pass
    same = torch.allclose(from_full, from_prefix, atol=1e-6)

    label = "abcdefgh"[:prefix_len]
    print(f"{label:<10}{prefix_len-1:<14}{str(same):<12}"
          f"full={from_full[0,:3].tolist()}  prefix={from_prefix[0,:3].tolist()}")


prefix    last-pos idx  identical?  sample values (first 3 dims)
a         0             True        full=[1.2655940055847168, 0.5423235297203064, -1.056722640991211]  prefix=[1.2655938863754272, 0.542323648929596, -1.0567227602005005]
ab        1             True        full=[-1.814671516418457, 0.24059291183948517, 0.2435184270143509]  prefix=[-1.8146713972091675, 0.24059295654296875, 0.24351856112480164]
abc       2             True        full=[-1.7201393842697144, 1.3883235454559326, 0.6254863142967224]  prefix=[-1.720139503479004, 1.3883236646652222, 0.6254865527153015]
abcd      3             True        full=[-2.6280617713928223, 1.356939435005188, 0.7863212823867798]  prefix=[-2.6280617713928223, 1.356939435005188, 0.7863212823867798]
abcde     4             True        full=[0.238181933760643, -0.2198280245065689, 0.7137393951416016]  prefix=[0.238181933760643, -0.2198280245065689, 0.7137393951416016]
abcdef    5             True        full=[-0.1150452047586441, 1.0679932832

In [67]:
full_out.shape, full_sequence.shape

(torch.Size([1, 8, 16]), torch.Size([1, 8, 16]))